In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [103]:
url_listing_berlin = "https://raw.githubusercontent.com/Nill2nn/Statistical-Programming/refs/heads/Nill2nn-patch-excel/listings_berlin.csv"
listings_berlin = pd.read_csv(url_listing_berlin)

In [104]:
def clean_price(price_str):
    if pd.isna(price_str):
        return None
    if isinstance(price_str, str):
        return float(price_str.replace('$', '').replace(',', ''))
    return float(price_str)

listings_berlin['price'] = listings_berlin['price'].apply(clean_price)

In [105]:
def clean_baths(bathroom):
    if pd.isna(bathroom):
        return None

    bathroom = str(bathroom).lower()

    if 'half' in bathroom:
        return 0.5
    else:
        numpart = bathroom.split(' ')[0]
        return float(numpart)


listings_berlin['bathrooms_text'] = listings_berlin['bathrooms_text'].apply(clean_baths)

In [106]:
#Task1
#we cleaned the data to work with numeric numbers
#we remove 0 and not defined values in price column
listings_berlin = listings_berlin.dropna(subset=['price','bathrooms_text', 'bedrooms'])
listings_berlin = listings_berlin[listings_berlin['price'] > 0]


In [107]:
#we define target matrix - price log; and matrix of our features consisting of numeric values and 2 dummy values
Y = np.log(listings_berlin['price'])
X = listings_berlin[["bedrooms", "bathrooms_text", "availability_365", "accommodates", "instant_bookable", "room_type"]]


In [108]:
#we create dummy variables 
X = pd.get_dummies(X, columns=['instant_bookable', 'room_type'], drop_first=True).astype(float)
X.isna().sum()


bedrooms                  0
bathrooms_text            0
availability_365          0
accommodates              0
instant_bookable_t        0
room_type_Hotel room      0
room_type_Private room    0
room_type_Shared room     0
dtype: int64

In [109]:
print(f'Sample size: {len(X)} observations')
X['instant_bookable_t'].unique()
print(X.dtypes)

Sample size: 9203 observations
bedrooms                  float64
bathrooms_text            float64
availability_365          float64
accommodates              float64
instant_bookable_t        float64
room_type_Hotel room      float64
room_type_Private room    float64
room_type_Shared room     float64
dtype: object


In [110]:
'''Following numeric variables were included in the model as features:
1. Bedrooms - number of bedrooms directly affects the price of the listing. More bedrooms typically mean a higher price.
2. Bathrooms_text - the number of bathrooms also influences the price. More bathrooms can increase the price of the listing.
3. Accommodates - This variable measures the maximum number of guests that can stay in the listing. Higher capacity usually leads to higher prices.
4. Review scores rating - Higher ratings indicate better perceived quality of the listing, which may allow hosts to charge higher prices.
5. Reviews_per_months - This value shows the popularity of listing and demand on monthly basis.
6. Availability_365 - This measures how often the property is available during the year.
Two categorical variables were included as dummy variables:
7. Room_type - Price differs significantly depending on whether the listing is an entire home, private room, or shared room
8. Instant bookable: Listings that allow instant booking may attract more demand and therefore affect pricing. '''

'Following numeric variables were included in the model as features:\n1. Bedrooms - number of bedrooms directly affects the price of the listing. More bedrooms typically mean a higher price.\n2. Bathrooms_text - the number of bathrooms also influences the price. More bathrooms can increase the price of the listing.\n3. Accommodates - This variable measures the maximum number of guests that can stay in the listing. Higher capacity usually leads to higher prices.\n4. Review scores rating - Higher ratings indicate better perceived quality of the listing, which may allow hosts to charge higher prices.\n5. Reviews_per_months - This value shows the popularity of listing and demand on monthly basis.\n6. Availability_365 - This measures how often the property is available during the year.\nTwo categorical variables were included as dummy variables:\n7. Room_type - Price differs significantly depending on whether the listing is an entire home, private room, or shared room\n8. Instant bookable: 

In [111]:
#Task2
Xc = sm.add_constant(X)
ols = sm.OLS(Y, Xc)
ols = ols.fit(cov_type = 'HC3')
ols.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.404
Model:                            OLS   Adj. R-squared:                  0.403
Method:                 Least Squares   F-statistic:                     698.9
Date:                Sat, 14 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:55:44   Log-Likelihood:                -6777.8
No. Observations:                9203   AIC:                         1.357e+04
Df Residuals:                    9194   BIC:                         1.364e+04
Df Model:                           8                                         
Covariance Type:                  HC3                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      4.1214      0.021    194.571      0.000       4.080       4.163
bedrooms                   0.0954      0.012      8.241      0.000       0.073       0.118
bathrooms_text             0.1069      0.019      5.634      0.000       0.070       0.144
availability_365       -7.506e-05   4.37e-05     -1.718      0.086      -0.000    1.06e-05
accommodates               0.0992      0.005     19.507      0.000       0.089       0.109
instant_bookable_t         0.2405      0.012     19.921      0.000       0.217       0.264
room_type_Hotel room       0.6737      0.128      5.253      0.000       0.422       0.925
room_type_Private room    -0.3414      0.014    -24.839      0.000      -0.368      -0.314
room_type_Shared room     -1.1156      0.093    -12.000      0.000      -1.298      -0.933
==============================================================================
Omnibus:                     3643.789   Durbin-Watson:                   1.609
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            63199.931
Skew:                           1.445   Prob(JB):                         0.00
Kurtosis:                      15.508   Cond. No.                     2.66e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
[2] The condition number is large, 2.66e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""